# Operational Clocks v0.7 — Standard-QM Null Model
Two qubit clocks with measurable phase. We test whether merger/separation changes relational clock readings beyond the standard Hamiltonian prediction. It should not: this notebook defines the null model that any new-time hypothesis must beat.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
I=np.eye(2,dtype=complex); X=np.array([[0,1],[1,0]],complex); Y=np.array([[0,-1j],[1j,0]],complex); Z=np.array([[1,0],[0,-1]],complex)
XI=np.kron(X,I); YI=np.kron(Y,I); ZI=np.kron(Z,I); IX=np.kron(I,X); IY=np.kron(I,Y); IZ=np.kron(I,Z); ZZ=np.kron(Z,Z)
plus=np.array([1,1],complex)/np.sqrt(2); psi0=np.kron(plus,plus)
wA=1.0; wB=1.35
def bridge(t):
    on=1/(1+np.exp(-10*(t-3))); off=1/(1+np.exp(10*(t-7))); return .35*on*off
def H(t): return .5*wA*ZI+.5*wB*IZ+bridge(t)*ZZ
def step(psi,t,dt):
    e,v=np.linalg.eigh(H(t+dt/2)); return (v*np.exp(-1j*e*dt))@v.conj().T@psi
def phase(psi,OX,OY): return np.arctan2(np.real(np.vdot(psi,OY@psi)),np.real(np.vdot(psi,OX@psi)))


## Evolve measurable clock phases

In [ ]:
dt=.002; ts=np.arange(0,10+dt,dt); psi=psi0.copy(); pa=[]; pb=[]; g=[]
for k,t in enumerate(ts):
    pa.append(phase(psi,XI,YI)); pb.append(phase(psi,IX,IY)); g.append(bridge(t))
    if k<len(ts)-1: psi=step(psi,t,dt)
pa=np.unwrap(pa); pb=np.unwrap(pb); g=np.array(g)
ra=np.gradient(pa,ts); rb=np.gradient(pb,ts)
plt.figure(figsize=(9,4.5)); plt.plot(ts,ra,label='observed phase rate A'); plt.plot(ts,rb,label='observed phase rate B'); plt.plot(ts,g,label='coupling g(t)'); plt.legend(); plt.xlabel('t'); plt.grid(alpha=.25); plt.show()


The local Bloch-vector phase is an operational clock reading only while transverse coherence is nonzero. Interaction can reduce visibility and distort instantaneous phase-rate extraction; that is standard clock physics, not a new time dimension.

## Relational clock ratio and standard prediction

In [ ]:
mask=(np.abs(rb)>.15); R=np.full_like(ra,np.nan); R[mask]=ra[mask]/rb[mask]
R0=wA/wB
plt.figure(figsize=(9,4.5)); plt.plot(ts,R,label='measured phase-rate ratio'); plt.axhline(R0,ls='--',label='uncoupled ratio wA/wB'); plt.ylim(-2,2); plt.legend(); plt.xlabel('t'); plt.grid(alpha=.25); plt.show()
print('uncoupled R0=',R0)


## Exact null-model residual
A valid new-physics residual must compare data with the *full interacting standard model*, not with the uncoupled ratio. Here the same Hamiltonian generates the observation and the null prediction, so the residual should vanish up to numerical error.

In [ ]:
# Re-run the same declared standard Hamiltonian as the null prediction
psi=psi0.copy(); qa=[]; qb=[]
for k,t in enumerate(ts):
    qa.append(phase(psi,XI,YI)); qb.append(phase(psi,IX,IY))
    if k<len(ts)-1: psi=step(psi,t,dt)
qa=np.unwrap(qa); qb=np.unwrap(qb); qra=np.gradient(qa,ts); qrb=np.gradient(qb,ts)
Rstd=np.full_like(qra,np.nan); m=np.abs(qrb)>.15; Rstd[m]=qra[m]/qrb[m]
res=R-Rstd
print('max |Delta R|=',np.nanmax(np.abs(res)))


## Scientific boundary
A change relative to the uncoupled ratio wA/wB is not evidence for new time: ordinary interaction already changes clock observables. The relevant quantity is Delta R = R_observed - R_standard(full Hamiltonian + environment + GR). In this null notebook Delta R is zero by construction. v0.8 may introduce a clearly labeled phenomenological temporal term only to determine what an actually falsifiable signal would look like.